# BoL Measurement Tools — RunPod Jupyter

Run **Cell 1** (install + config), then **Cell 2** (load model), then any measurement cell in any order.

| Cell | Tool | Purpose |
|------|------|---------|
| 1 | Install + Config | Setup — edit MODEL/LABEL here |
| 2 | Load Model | Shared model load (run once) |
| 3 | Silhouette | Semantic separation by layer |
| 4 | Semantic Probe | Cluster validation |
| 5 | Layer Sweep | Layer importance via ablation |
| 6 | Component Ablation | Attention vs MLP importance |
| 7 | Gradient Attribution | Gradient-based layer importance |
| 8 | Compression Map | Quantization tolerance per layer |
| 9 | CKA | Layer similarity analysis |
| 10 | Attention Map | Attention pattern analysis |
| 11 | Weight Fingerprint | Weight statistics |
| 12 | Random Baseline | Null hypothesis control |
| 13 | Multirun Stability | Reproducibility test |

Outputs save to `/workspace/bol_results/`.

In [ ]:
# ── CELL 1: INSTALL + CONFIG ─────────────────────────────────────────────────
import subprocess
# Uninstall torchvision to avoid operator conflict with this version of torch/transformers
subprocess.run(['pip', 'uninstall', '-y', 'torchvision'], capture_output=True)
subprocess.run(['pip', 'install', '-q', 'transformers', 'torch', 'scipy', 'numpy', 'peft'], check=True)

import torch
import torch.nn.functional as F
import json
import numpy as np
import os
import math
import hashlib
from scipy import stats as scipy_stats
from transformers import AutoModelForCausalLM, AutoTokenizer

# ── EDIT THESE TO SWITCH MODELS ───────────────────────────────────────────────
MODEL      = '/workspace/bella-v5-solid'
TOKENIZER  = 'meta-llama/Llama-3.2-3B-Instruct'   # tokenizer (separate from model weights)
LABEL      = 'bella-v5-solid'
OUTPUT_DIR = '/workspace/bol_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.float32

EVAL_TEXTS = [
    'The quick brown fox jumps over the lazy dog',
    'In the beginning was the word and the word was with God',
    'Neural networks learn hierarchical representations of data',
    'The population of Tokyo is approximately 14 million people',
    'Water boils at 100 degrees Celsius at standard atmospheric pressure',
]
PROBES = [
    'The capital of France is',
    '2 + 2 =',
    'Once upon a time there was',
    'The meaning of life is',
    'def fibonacci(n):',
]
RELATED_PAIRS = [
    ('dog', 'wolf'), ('cat', 'lion'), ('river', 'ocean'),
    ('happy', 'joyful'), ('cold', 'frozen'), ('king', 'queen'),
    ('python', 'code'), ('car', 'vehicle'), ('run', 'sprint'),
]
UNRELATED_PAIRS = [
    ('dog', 'democracy'), ('cat', 'algebra'), ('river', 'justice'),
    ('happy', 'concrete'), ('cold', 'philosophy'), ('king', 'equation'),
    ('python', 'banana'), ('car', 'silence'), ('run', 'orange'),
]
CLUSTER_WORDS = [
    'dog', 'wolf', 'puppy', 'cat', 'kitten', 'tiger',
    'car', 'truck', 'vehicle', 'banana', 'apple', 'fruit',
    'happy', 'joy', 'sadness', 'code', 'python', 'programming',
    'silence', 'peace', 'war',
]
CLUSTERS = {
    'animals':  ['dog', 'wolf', 'puppy'],
    'felines':  ['cat', 'kitten', 'tiger'],
    'vehicles': ['car', 'truck', 'vehicle'],
    'food':     ['banana', 'apple', 'fruit'],
    'emotions': ['happy', 'joy', 'sadness'],
    'tech':     ['code', 'python', 'programming'],
    'abstract': ['silence', 'peace', 'war'],
}
NOISE_LEVELS = {'int8': 0.004, 'int6': 0.016, 'int4': 0.063, 'int2': 0.250}
COMPRESSION_THRESHOLD = 0.5

print(f'Config ready | Model: {MODEL} | Tokenizer: {TOKENIZER} | Device: {DEVICE}')

In [ ]:
# ── CELL 2: LOAD SHARED MODEL ─────────────────────────────────────────────────
# Run once. All measurement cells below reuse this model object.
# Tokenizer loaded from Llama-3.2-3B-Instruct; weights loaded from MODEL path.
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    dtype=DTYPE,
    device_map=DEVICE,
    output_hidden_states=True,
    low_cpu_mem_usage=True,
)
model.eval()
N          = model.config.num_hidden_layers
NUM_HEADS  = model.config.num_attention_heads
VOCAB_SIZE = model.config.vocab_size

print(f'Loaded: {MODEL}')
print(f'Tokenizer: {TOKENIZER}')
print(f'Layers: {N} | Heads: {NUM_HEADS} | Vocab: {VOCAB_SIZE} | Device: {DEVICE}')

In [ ]:
# ── CELL 3: SILHOUETTE — Semantic Separation Map ─────────────────────────────
def run_silhouette():
    def get_layers(word):
        inp = tokenizer(word, return_tensors='pt')
        if DEVICE == 'cuda':
            inp = {k: v.to(DEVICE) for k, v in inp.items()}
        with torch.no_grad():
            out = model(**inp)
        return [h[0, -1, :].cpu().float().numpy() for h in out.hidden_states]

    print('Extracting hidden states...')
    all_words = list({w for pair in RELATED_PAIRS + UNRELATED_PAIRS for w in pair})
    cache = {w: get_layers(w) for w in all_words}
    print(f'  Cached {len(cache)} words x {N+1} layers')
    print()

    print(f'{"Layer":>6}  {"rel avg":>9}  {"unrel avg":>10}  {"sep":>8}  {"p-val":>8}  bar')
    print('-' * 75)

    best_layer, best_sep = 0, -999
    layer_data = []

    for layer in range(N + 1):
        rel_sims = [
            F.cosine_similarity(
                torch.tensor(cache[a][layer]).unsqueeze(0),
                torch.tensor(cache[b][layer]).unsqueeze(0)
            ).item() for a, b in RELATED_PAIRS
        ]
        unrel_sims = [
            F.cosine_similarity(
                torch.tensor(cache[a][layer]).unsqueeze(0),
                torch.tensor(cache[b][layer]).unsqueeze(0)
            ).item() for a, b in UNRELATED_PAIRS
        ]
        ra, ua = np.mean(rel_sims), np.mean(unrel_sims)
        sep = ra - ua
        try:
            _, pval = scipy_stats.mannwhitneyu(rel_sims, unrel_sims, alternative='greater')
        except ValueError:
            pval = 1.0
        bar = '█' * int(max(sep, 0) * 50)
        marker = ' <- BEST' if sep > best_sep else ''
        if sep > best_sep:
            best_sep, best_layer = sep, layer
        sig = '*' if pval < 0.05 else ' '
        print(f'  L{layer:02d}   {ra:>9.4f}  {ua:>10.4f}  {sep:>+8.4f}  {pval:>7.4f}{sig}  {bar}{marker}')
        layer_data.append({
            'layer': layer, 'rel_avg': round(ra, 4), 'unrel_avg': round(ua, 4),
            'separation': round(sep, 4), 'p_value': round(pval, 6),
            'significant': bool(pval < 0.05),
            'rel_sims': [round(s, 4) for s in rel_sims],
            'unrel_sims': [round(s, 4) for s in unrel_sims],
        })

    ranked = sorted(layer_data, key=lambda x: x['separation'], reverse=True)
    print()
    print('=' * 75)
    print('TOP 10 LAYERS BY SEMANTIC SEPARATION')
    print('=' * 75)
    for r in ranked[:10]:
        bar = '█' * int(max(r['separation'], 0) * 50)
        sig_tag = ' (p<0.05)' if r['significant'] else ''
        print(f'  Layer {r["layer"]:02d}  sep={r["separation"]:+.4f}  {bar}{sig_tag}')
    print(f'-> Best layer: {best_layer} (sep={best_sep:.4f})')
    print(f'-> These are your load-bearing walls.')

    out_file = f'{OUTPUT_DIR}/bol_silhouette_{LABEL}.json'
    with open(out_file, 'w') as f:
        json.dump({
            'label': LABEL, 'model': MODEL, 'device': DEVICE, 'dtype': str(DTYPE),
            'num_layers': N, 'best_layer': best_layer, 'best_separation': round(best_sep, 4),
            'layer_data': layer_data, 'ranked_top_10': ranked[:10],
        }, f, indent=2)
    print(f'Saved: {out_file}')

run_silhouette()

In [ ]:
# ── CELL 4: SEMANTIC PROBE — Cluster Validation ───────────────────────────────
def run_semantic_probe():
    LAYERS = [0, 5, 10, 15, 20, 25, 29, 30]
    LAYERS = [l for l in LAYERS if l <= N]

    def get_hidden(word, layer_idx):
        inputs = tokenizer(word, return_tensors='pt')
        if DEVICE == 'cuda':
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        hidden = outputs.hidden_states[layer_idx]
        return hidden.mean(dim=1).squeeze().cpu().float()

    def cos(a, b):
        return F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

    print(f'Extracting hidden states for {len(CLUSTER_WORDS)} words across {len(LAYERS)} layers...')
    vectors = {layer: {w: get_hidden(w, layer) for w in CLUSTER_WORDS} for layer in LAYERS}
    print('  Done.')
    print()

    display_layer = LAYERS[-1]
    print(f'=== SIMILARITY MATRIX (Layer {display_layer}) ===')
    print(f'{"":15}', end='')
    for w in CLUSTER_WORDS:
        print(f'{w:>10}', end='')
    print()
    for w1 in CLUSTER_WORDS:
        print(f'{w1:15}', end='')
        for w2 in CLUSTER_WORDS:
            sim = cos(vectors[display_layer][w1], vectors[display_layer][w2])
            print(f'{sim:10.3f}', end='')
        print()

    print()
    print('=' * 85)
    print('CLUSTER VALIDATION BY LAYER')
    print('=' * 85)
    print(f'{"Layer":>6}  {"Cluster":<12}  {"within":>8}  {"cross":>9}  {"sep":>8}  {"p-val":>8}  {"verdict":<15}')
    print('-' * 85)

    layer_results = []
    for layer in LAYERS:
        layer_data = {'layer': layer, 'clusters': {}}
        for cname, cwords in CLUSTERS.items():
            within = [
                cos(vectors[layer][w1], vectors[layer][w2])
                for i, w1 in enumerate(cwords) for w2 in cwords[i+1:]
            ]
            cross_words = [w for cn, cw in CLUSTERS.items() if cn != cname for w in cw]
            cross = [
                cos(vectors[layer][w1], vectors[layer][w2])
                for w1 in cwords for w2 in cross_words
            ]
            wa = np.mean(within) if within else 0
            ca = np.mean(cross) if cross else 0
            sep = wa - ca
            try:
                _, pval = scipy_stats.mannwhitneyu(within, cross, alternative='greater')
            except ValueError:
                pval = 1.0
            verdict = 'CLUSTERS' if sep > 0.05 and pval < 0.05 else 'WEAK' if sep > 0 else 'NO CLUSTER'
            sig = '*' if pval < 0.05 else ''
            print(f'  L{layer:02d}   {cname:<12}  {wa:>8.3f}  {ca:>9.3f}  {sep:>+8.3f}  {pval:>7.4f}{sig} {verdict:<15}')
            layer_data['clusters'][cname] = {
                'within_avg': round(wa, 4), 'cross_avg': round(ca, 4),
                'separation': round(sep, 4), 'p_value': round(pval, 6),
                'significant': pval < 0.05, 'verdict': verdict,
            }
        layer_results.append(layer_data)

    print()
    print('=' * 85)
    print('BEST LAYER FOR SEMANTIC CLUSTERING')
    print('=' * 85)
    layer_avg_sep = [(lr['layer'], np.mean([c['separation'] for c in lr['clusters'].values()])) for lr in layer_results]
    best_layer, best_sep = max(layer_avg_sep, key=lambda x: x[1])
    print(f'  Best layer: L{best_layer} (avg separation: {best_sep:+.4f})')

    out_file = f'{OUTPUT_DIR}/bol_semantic_probe_{LABEL}.json'
    with open(out_file, 'w') as f:
        json.dump({
            'label': LABEL, 'model': MODEL, 'device': DEVICE, 'dtype': str(DTYPE),
            'layers_probed': LAYERS, 'best_layer': int(best_layer),
            'best_avg_separation': round(float(best_sep), 4),
            'layer_results': layer_results,
        }, f, indent=2)
    print(f'Saved: {out_file}')

run_semantic_probe()

In [ ]:
# ── CELL 5: LAYER SWEEP — Layer Importance Ranking ───────────────────────────
def run_layer_sweep():
    tokenized = tokenizer(EVAL_TEXTS, return_tensors='pt', padding=True, truncation=True)
    labels = tokenized['input_ids'].clone()
    labels[labels == tokenizer.pad_token_id] = -100
    if DEVICE == 'cuda':
        tokenized = {k: v.to(DEVICE) for k, v in tokenized.items()}
        labels = labels.to(DEVICE)

    probe_prompt = 'The capital of France is'

    def get_batched_loss():
        with torch.no_grad():
            outputs = model(**tokenized, labels=labels)
        loss = outputs.loss
        if torch.isnan(loss) or torch.isinf(loss):
            return float('nan')
        return loss.item()

    def talk(prompt, max_new=40):
        inputs = tokenizer(prompt, return_tensors='pt')
        if DEVICE == 'cuda':
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new, do_sample=False)
        return tokenizer.decode(out[0], skip_special_tokens=True)

    print('Saving original weight snapshot (FP32)...')
    original_weights = {name: param.data.clone() for name, param in model.named_parameters()}

    def verify_integrity():
        hasher = hashlib.sha256()
        for name, param in model.named_parameters():
            if name in original_weights:
                hasher.update(param.data.cpu().numpy().tobytes())
        cur = hasher.hexdigest()
        horig = hashlib.sha256()
        for t in original_weights.values():
            horig.update(t.cpu().numpy().tobytes())
        orig = horig.hexdigest()
        return cur == orig, orig[:16], cur[:16]

    baseline_loss = get_batched_loss()
    if math.isnan(baseline_loss) or math.isinf(baseline_loss):
        raise RuntimeError('Baseline loss is NaN/Inf. Check model loading.')

    baseline_out = talk(probe_prompt)
    print(f'BASELINE loss: {baseline_loss:.4f}')
    print(f'BASELINE out:  {baseline_out[:80]}')
    print()
    print(f'{"Layer":>7}  {"Loss":>8}  {"Damage":>8}  {"Tag":<12}  Sample output')
    print('-' * 90)

    results = []
    for i in range(N):
        zeroed_params = []
        try:
            for name, param in model.named_parameters():
                if f'.layers.{i}.' in name and param.ndim >= 2:
                    param.data.zero_()
                    zeroed_params.append(name)
            if DEVICE == 'cuda':
                torch.cuda.synchronize()
            loss = get_batched_loss()
            if math.isnan(loss) or math.isinf(loss):
                loss = 999.9999
                damage, sample, tag, chg = 999.9999, 'NaN ERROR', 'CRITICAL', 'NaN'
            else:
                damage = loss - baseline_loss
                sample = talk(probe_prompt)
                changed = sample != baseline_out
                tag = 'CRITICAL' if damage > 1.0 else 'MODERATE' if damage > 0.3 else 'REMOVABLE'
                chg = 'CHANGED' if changed else 'stable'
            print(f'  L{i:02d}    {loss:>8.4f}  {damage:>+8.4f}  {tag:<12}  [{chg}] {sample[:50]}')
            results.append({'layer': i, 'loss': round(loss, 4), 'damage': round(damage, 4),
                            'changed': chg == 'CHANGED', 'sample': sample[:120]})
        finally:
            for name in zeroed_params:
                model.get_parameter(name).data.copy_(original_weights[name])
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()

    is_intact, orig_hash, final_hash = verify_integrity()
    print()
    print('=' * 70)
    if is_intact:
        print(f'INTEGRITY VERIFIED (Hash: {final_hash})')
    else:
        print(f'INTEGRITY FAIL! orig={orig_hash} final={final_hash}')
    print('=' * 70)

    ranked = sorted(results, key=lambda x: x['damage'], reverse=True)
    print()
    print('LAYER IMPORTANCE RANKING  (most critical -> safest to remove)')
    print('=' * 70)
    for rank, r in enumerate(ranked, 1):
        tag = 'CRITICAL' if r['damage'] > 1.0 else 'MODERATE' if r['damage'] > 0.3 else 'REMOVABLE'
        print(f'  #{rank:02d}  Layer {r["layer"]:02d}  damage: {r["damage"]:+.4f}  {tag}')

    out_file = f'{OUTPUT_DIR}/layer_importance_{LABEL}.json'
    with open(out_file, 'w') as f:
        json.dump({'baseline_loss': baseline_loss, 'ranked': ranked, 'by_layer': results}, f, indent=2)
    print(f'Saved: {out_file}')

run_layer_sweep()

In [ ]:
# ── CELL 6: COMPONENT ABLATION — Attention vs MLP Importance ─────────────────
def run_component_ablation():
    COMPONENTS = [
        'self_attn.q_proj', 'self_attn.k_proj', 'self_attn.v_proj', 'self_attn.o_proj',
        'mlp.gate_proj', 'mlp.up_proj', 'mlp.down_proj',
    ]

    tokenized = tokenizer(EVAL_TEXTS, return_tensors='pt', padding=True, truncation=True)
    labels = tokenized['input_ids'].clone()
    labels[labels == tokenizer.pad_token_id] = -100
    if DEVICE == 'cuda':
        tokenized = {k: v.to(DEVICE) for k, v in tokenized.items()}
        labels = labels.to(DEVICE)

    def get_loss():
        with torch.no_grad():
            outputs = model(**tokenized, labels=labels)
        loss = outputs.loss
        return float('nan') if torch.isnan(loss) or torch.isinf(loss) else loss.item()

    def talk(p, max_new=40):
        inp = tokenizer(p, return_tensors='pt')
        if DEVICE == 'cuda':
            inp = {k: v.to(DEVICE) for k, v in inp.items()}
        with torch.no_grad():
            out = model.generate(**inp, max_new_tokens=max_new, do_sample=False)
        return tokenizer.decode(out[0], skip_special_tokens=True)

    def verify_integrity(original_weights):
        hasher = hashlib.sha256()
        for name, param in model.named_parameters():
            if name in original_weights:
                hasher.update(param.data.cpu().numpy().tobytes())
        cur = hasher.hexdigest()
        horig = hashlib.sha256()
        for t in original_weights.values():
            horig.update(t.cpu().numpy().tobytes())
        orig = horig.hexdigest()
        return cur == orig, orig[:16], cur[:16]

    print('Saving original weight snapshot (FP32)...')
    original_weights = {name: param.data.clone() for name, param in model.named_parameters()}

    baseline_loss = get_loss()
    if math.isnan(baseline_loss) or math.isinf(baseline_loss):
        raise RuntimeError('Baseline loss is NaN/Inf.')
    baseline_outputs = {p: talk(p) for p in PROBES}
    print(f'BASELINE loss: {baseline_loss:.4f}')
    print()

    results = []
    for comp in COMPONENTS:
        zeroed_params = []
        try:
            for name, param in model.named_parameters():
                if comp in name and param.ndim >= 2:
                    param.data.zero_()
                    zeroed_params.append(name)
            if DEVICE == 'cuda':
                torch.cuda.synchronize()
            loss = get_loss()
            if math.isnan(loss) or math.isinf(loss):
                loss_val, delta = 999.9999, 999.9999
            else:
                loss_val, delta = loss, loss - baseline_loss
            print(f'=== ZEROED: {comp}  ({len(zeroed_params)} tensors)  loss: {loss_val:.4f}  (delta {delta:+.4f}) ===')
            row = {'component': comp, 'zeroed_tensors': len(zeroed_params),
                   'loss': round(loss_val, 4), 'damage': round(delta, 4), 'outputs': {}}
            for p in PROBES:
                out = talk(p)
                changed = out != baseline_outputs[p]
                tag = 'CHANGED' if changed else 'stable'
                print(f'  [{tag:7s}] {p[:35]:<35} -> {out[:55]}')
                row['outputs'][p] = {'text': out, 'changed': changed}
            results.append(row)
        finally:
            for name in zeroed_params:
                model.get_parameter(name).data.copy_(original_weights[name])
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()

    is_intact, orig_hash, final_hash = verify_integrity(original_weights)
    print()
    print('=' * 60)
    if is_intact:
        print(f'INTEGRITY VERIFIED (Hash: {final_hash})')
    else:
        print(f'INTEGRITY FAIL! orig={orig_hash} final={final_hash}')

    results.sort(key=lambda x: x['damage'], reverse=True)
    print()
    print('COMPONENT IMPORTANCE RANKING')
    print('=' * 60)
    for r in results:
        tag = 'CRITICAL' if r['damage'] > 1.0 else 'MODERATE' if r['damage'] > 0.3 else 'REMOVABLE'
        print(f'  {r["component"]:<30}  delta_loss: {r["damage"]:+.4f}  {tag}')

    out_file = f'{OUTPUT_DIR}/bol_component_report_{LABEL}.json'
    with open(out_file, 'w') as f:
        json.dump({'label': LABEL, 'model': MODEL, 'baseline_loss': baseline_loss,
                   'results': results}, f, indent=2)
    print(f'Saved: {out_file}')

run_component_ablation()

In [ ]:
# ── CELL 7: GRADIENT ATTRIBUTION — Gradient-Based Layer Importance ────────────
def run_gradient_attribution():
    tokenized = tokenizer(EVAL_TEXTS, return_tensors='pt', padding=True, truncation=True)
    labels = tokenized['input_ids'].clone()
    labels[labels == tokenizer.pad_token_id] = -100
    if DEVICE == 'cuda':
        tokenized = {k: v.to(DEVICE) for k, v in tokenized.items()}
        labels = labels.to(DEVICE)

    print('=' * 75)
    print('TEST 1: GRADIENT MAGNITUDE PER LAYER')
    print('(Higher = more important for loss)')
    print('=' * 75)

    model.zero_grad()
    outputs = model(**tokenized, labels=labels)
    loss = outputs.loss
    loss.backward()
    print(f'Loss: {loss.item():.4f}')
    print()
    print(f'{"Layer":>6}  {"grad_norm":>11}  {"grad_mean":>11}  {"param_norm":>11}  {"ratio":>8}  bar')
    print('-' * 75)

    layer_grad_stats = []
    for i in range(N):
        grad_norms, param_norms, grad_means = [], [], []
        for name, param in model.named_parameters():
            if f'.layers.{i}.' not in name or param.grad is None or param.ndim < 2:
                continue
            g = param.grad.float()
            grad_norms.append(g.norm().item())
            grad_means.append(g.abs().mean().item())
            param_norms.append(param.data.float().norm().item())
        if not grad_norms:
            continue
        total_grad_norm = np.sqrt(sum(g ** 2 for g in grad_norms))
        total_param_norm = np.sqrt(sum(p ** 2 for p in param_norms))
        ratio = total_grad_norm / (total_param_norm + 1e-10)
        avg_grad_mean = np.mean(grad_means)
        bar = '█' * min(int(ratio * 500), 40)
        print(f'  L{i:02d}   {total_grad_norm:>11.6f}  {avg_grad_mean:>11.8f}  {total_param_norm:>11.2f}  {ratio:>8.6f}  {bar}')
        layer_grad_stats.append({
            'layer': i, 'gradient_norm': round(total_grad_norm, 6),
            'gradient_mean': round(avg_grad_mean, 8), 'parameter_norm': round(total_param_norm, 4),
            'gradient_to_param_ratio': round(ratio, 8),
        })

    print()
    print('=' * 75)
    print('TEST 2: COMPONENT-LEVEL GRADIENT ATTRIBUTION')
    print('=' * 75)
    component_types = [
        'self_attn.q_proj', 'self_attn.k_proj', 'self_attn.v_proj', 'self_attn.o_proj',
        'mlp.gate_proj', 'mlp.up_proj', 'mlp.down_proj',
    ]
    print(f'  {"Component":<25}  {"total_grad":>12}  {"avg_per_layer":>14}  {"% of total":>11}')
    print('-' * 70)
    component_grads = {}
    total_grad = 0
    for comp in component_types:
        grad_sum, count = 0, 0
        for name, param in model.named_parameters():
            if comp not in name or param.grad is None or param.ndim < 2:
                continue
            grad_sum += param.grad.float().norm().item() ** 2
            count += 1
        grad_sum = np.sqrt(grad_sum)
        total_grad += grad_sum
        component_grads[comp] = {'total': grad_sum, 'count': count}
    for comp in component_types:
        cg = component_grads[comp]
        pct = (cg['total'] / total_grad * 100) if total_grad > 0 else 0
        avg = cg['total'] / cg['count'] if cg['count'] > 0 else 0
        print(f'  {comp:<23}  {cg["total"]:>12.6f}  {avg:>14.6f}  {pct:>10.1f}%')

    print()
    print('=' * 75)
    print('TEST 3: GRADIENT FLOW THROUGH LAYERS')
    print('=' * 75)
    gradient_flow = {'pattern': 'INSUFFICIENT_DATA'}
    if len(layer_grad_stats) >= 2:
        norms = [s['gradient_norm'] for s in layer_grad_stats]
        max_norm = max(norms)
        min_norm = min(n for n in norms if n > 0) if any(n > 0 for n in norms) else 1e-10
        flow_ratio = max_norm / min_norm if min_norm > 0 else float('inf')
        first_half = np.mean(norms[:len(norms)//2])
        second_half = np.mean(norms[len(norms)//2:])
        pattern = ('SEVERE GRADIENT IMBALANCE' if flow_ratio > 100
                   else 'MODERATE GRADIENT IMBALANCE' if flow_ratio > 10
                   else 'HEALTHY GRADIENT FLOW')
        direction = ('VANISHING (early layers starved)' if first_half < second_half * 0.5
                     else 'EXPLODING (early layers oversaturated)' if first_half > second_half * 2
                     else 'BALANCED')
        print(f'  Max/min gradient ratio: {flow_ratio:.2f}x')
        print(f'  Early vs late layer avg: {first_half:.6f} vs {second_half:.6f}')
        print(f'  Pattern: {pattern}')
        print(f'  Direction: {direction}')
        gradient_flow = {'max_min_ratio': round(flow_ratio, 4), 'pattern': pattern, 'direction': direction}

    ranked = sorted(layer_grad_stats, key=lambda x: x['gradient_to_param_ratio'], reverse=True)
    print()
    print('LAYER IMPORTANCE RANKING (by gradient-to-parameter ratio)')
    print('=' * 75)
    for rank, r in enumerate(ranked, 1):
        bar = '█' * min(int(r['gradient_to_param_ratio'] * 500), 30)
        print(f'  #{rank:02d}  Layer {r["layer"]:02d}  ratio: {r["gradient_to_param_ratio"]:.6f}  {bar}')

    model.zero_grad()  # clean up

    out_file = f'{OUTPUT_DIR}/bol_gradient_attribution_{LABEL}.json'
    component_grads_json = {k: {'total_gradient_norm': round(v['total'], 6), 'layer_count': v['count']}
                            for k, v in component_grads.items()}
    with open(out_file, 'w') as f:
        json.dump({
            'label': LABEL, 'model': MODEL, 'num_layers': N, 'loss': round(loss.item(), 4),
            'layer_gradient_stats': layer_grad_stats,
            'layer_ranking': [{'rank': i+1, **r} for i, r in enumerate(ranked)],
            'component_attribution': component_grads_json,
            'gradient_flow': gradient_flow,
        }, f, indent=2)
    print(f'Saved: {out_file}')

run_gradient_attribution()

In [ ]:
# ── CELL 8: COMPRESSION MAP — Quantization Tolerance ─────────────────────────
def run_compression_map():
    eval_texts = [
        'The quick brown fox jumps over the lazy dog',
        'Water boils at 100 degrees Celsius',
        'Neural networks learn hierarchical representations',
    ]
    tokenized = tokenizer(eval_texts, return_tensors='pt', padding=True, truncation=True)
    labels = tokenized['input_ids'].clone()
    labels[labels == tokenizer.pad_token_id] = -100
    if DEVICE == 'cuda':
        tokenized = {k: v.to(DEVICE) for k, v in tokenized.items()}
        labels = labels.to(DEVICE)

    def get_batched_loss():
        with torch.no_grad():
            outputs = model(**tokenized, labels=labels)
        loss = outputs.loss
        if torch.isnan(loss) or torch.isinf(loss):
            return float('nan')
        return loss.item()

    def verify_integrity(original_weights):
        hasher = hashlib.sha256()
        for name, param in model.named_parameters():
            if name in original_weights:
                hasher.update(param.data.cpu().numpy().tobytes())
        cur = hasher.hexdigest()
        horig = hashlib.sha256()
        for t in original_weights.values():
            horig.update(t.cpu().numpy().tobytes())
        orig = horig.hexdigest()
        return cur == orig, orig[:16], cur[:16]

    print('Saving original weight snapshot (FP32)...')
    original_weights = {name: param.data.clone() for name, param in model.named_parameters()}

    baseline_loss = get_batched_loss()
    if math.isnan(baseline_loss) or math.isinf(baseline_loss):
        raise RuntimeError('Baseline loss is NaN/Inf.')
    print(f'Baseline loss: {baseline_loss:.4f}')
    print()
    print(f'{"Layer":>6}', end='')
    for ln in NOISE_LEVELS:
        print(f'  {ln:>8}', end='')
    print(f'  {"tolerance":>10}')
    print('-' * 65)

    layer_results = []
    for i in range(N):
        print(f'  L{i:02d}  ', end='', flush=True)
        row = {'layer': i, 'noise_deltas': {}, 'tolerance': 'fp16'}
        best_tolerance = 'fp16'
        layer_param_names = [
            name for name, param in model.named_parameters()
            if f'.layers.{i}.' in name and param.ndim >= 2
        ]
        for level_name, noise_scale in NOISE_LEVELS.items():
            try:
                for name in layer_param_names:
                    param = model.get_parameter(name)
                    orig_tensor = original_weights[name]
                    scale = orig_tensor.abs().mean()
                    noise = torch.randn_like(orig_tensor) * scale * noise_scale
                    if DEVICE == 'cuda':
                        noise = noise.to(DEVICE)
                    param.data.add_(noise)
                if DEVICE == 'cuda':
                    torch.cuda.synchronize()
                delta = get_batched_loss() - baseline_loss
                if math.isnan(delta) or math.isinf(delta):
                    delta = 999.9999
                row['noise_deltas'][level_name] = round(delta, 4)
                print(f'  {delta:>+8.4f}', end='', flush=True)
                if delta < COMPRESSION_THRESHOLD:
                    best_tolerance = level_name
            finally:
                for name in layer_param_names:
                    model.get_parameter(name).data.copy_(original_weights[name])
                if DEVICE == 'cuda':
                    torch.cuda.empty_cache()
        row['tolerance'] = best_tolerance
        tag = ('[compressible]' if 'int4' in best_tolerance or 'int2' in best_tolerance
               else '[moderate]' if 'int6' in best_tolerance
               else '[keep fp16]')
        print(f'  {tag} {best_tolerance:>8}')
        layer_results.append(row)

    is_intact, orig_hash, final_hash = verify_integrity(original_weights)
    print()
    print('=' * 65)
    if is_intact:
        print(f'INTEGRITY VERIFIED (Hash: {final_hash})')
    else:
        print(f'INTEGRITY FAIL! orig={orig_hash} final={final_hash}')
    print('=' * 65)
    print()
    print('COMPRESSION MAP  (compressible -> keep at fp16)')
    print('=' * 65)
    for r in layer_results:
        tag = ('compressible' if 'int4' in r['tolerance'] or 'int2' in r['tolerance']
               else 'moderate' if 'int6' in r['tolerance']
               else 'keep fp16')
        print(f'  Layer {r["layer"]:02d}: [{tag}] tolerates {r["tolerance"]}')

    out_file = f'{OUTPUT_DIR}/bol_compression_map_{LABEL}.json'
    with open(out_file, 'w') as f:
        json.dump({'label': LABEL, 'model': MODEL, 'baseline_loss': baseline_loss,
                   'layer_results': layer_results}, f, indent=2)
    print(f'Saved: {out_file}')

run_compression_map()

In [ ]:
# ── CELL 9: CKA — Centered Kernel Alignment ───────────────────────────────────
def run_cka():
    def centering_matrix(n):
        return np.eye(n) - np.ones((n, n)) / n

    def linear_cka(X, Y):
        n = X.shape[0]
        H = centering_matrix(n)
        K, L = X @ X.T, Y @ Y.T
        hsic_kl = np.trace(H @ K @ H @ L) / ((n - 1) ** 2)
        hsic_kk = np.trace(H @ K @ H @ K) / ((n - 1) ** 2)
        hsic_ll = np.trace(H @ L @ H @ L) / ((n - 1) ** 2)
        denom = np.sqrt(hsic_kk * hsic_ll)
        return 0.0 if denom < 1e-10 else hsic_kl / denom

    def get_hidden(word, layer_idx):
        inputs = tokenizer(word, return_tensors='pt')
        if DEVICE == 'cuda':
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        hidden = outputs.hidden_states[layer_idx]
        return hidden.mean(dim=1).squeeze().cpu().float().numpy()

    SAMPLE_LAYERS = list(range(0, N + 1, max(1, N // 10)))
    if N not in SAMPLE_LAYERS:
        SAMPLE_LAYERS.append(N)
    SAMPLE_LAYERS = sorted(set(SAMPLE_LAYERS))

    print(f'Extracting representations for {len(CLUSTER_WORDS)} words across {len(SAMPLE_LAYERS)} layers...')
    rep_matrices = {}
    for layer in SAMPLE_LAYERS:
        vecs = [get_hidden(w, layer) for w in CLUSTER_WORDS]
        rep_matrices[layer] = np.stack(vecs)
    print('  Done.')
    print()

    print('=' * 80)
    print('TEST 1: LAYER-TO-LAYER CKA SIMILARITY')
    print('=' * 80)
    print(f'{"":>8}', end='')
    for l in SAMPLE_LAYERS:
        print(f'  L{l:02d}', end='')
    print()
    cka_matrix = {}
    for li in SAMPLE_LAYERS:
        print(f'  L{li:02d}  ', end='')
        cka_matrix[li] = {}
        for lj in SAMPLE_LAYERS:
            cka_val = linear_cka(rep_matrices[li], rep_matrices[lj])
            cka_matrix[li][lj] = round(cka_val, 3)
            print(f' {cka_val:.2f}', end='')
        print()

    print()
    print('=' * 80)
    print('TEST 2: CLUSTER CKA ANALYSIS')
    print('=' * 80)
    print(f'{"Layer":>6}  {"within_CKA":>11}  {"cross_CKA":>10}  {"sep":>8}  verdict')
    print('-' * 55)

    cluster_cka_by_layer = []
    for layer in SAMPLE_LAYERS:
        cluster_reps = {}
        for cname, cwords in CLUSTERS.items():
            idxs = [CLUSTER_WORDS.index(w) for w in cwords if w in CLUSTER_WORDS]
            if len(idxs) >= 2:
                cluster_reps[cname] = rep_matrices[layer][idxs]
        within_ckas = []
        for cname, crep in cluster_reps.items():
            for i in range(crep.shape[0]):
                for j in range(i + 1, crep.shape[0]):
                    sim = np.dot(crep[i], crep[j]) / (np.linalg.norm(crep[i]) * np.linalg.norm(crep[j]) + 1e-10)
                    within_ckas.append(sim)
        cross_ckas = []
        items = list(cluster_reps.items())
        for ci in range(len(items)):
            for cj in range(ci + 1, len(items)):
                cr1, cr2 = items[ci][1], items[cj][1]
                for i in range(cr1.shape[0]):
                    for j in range(cr2.shape[0]):
                        sim = np.dot(cr1[i], cr2[j]) / (np.linalg.norm(cr1[i]) * np.linalg.norm(cr2[j]) + 1e-10)
                        cross_ckas.append(sim)
        within_avg = np.mean(within_ckas) if within_ckas else 0
        cross_avg = np.mean(cross_ckas) if cross_ckas else 0
        sep = within_avg - cross_avg
        verdict = 'CLUSTERED' if sep > 0.05 else 'WEAK' if sep > 0 else 'NO CLUSTER'
        print(f'  L{layer:02d}   {within_avg:>11.4f}  {cross_avg:>10.4f}  {sep:>+8.4f}  {verdict}')
        cluster_cka_by_layer.append({'layer': layer, 'within_cka': round(within_avg, 4),
                                      'cross_cka': round(cross_avg, 4), 'separation': round(sep, 4),
                                      'verdict': verdict})

    print()
    print('=' * 80)
    print('TEST 3: REPRESENTATION EVOLUTION (CKA vs Layer 0)')
    print('=' * 80)
    evolution = []
    transform_point = None
    for layer in SAMPLE_LAYERS:
        cka_val = linear_cka(rep_matrices[0], rep_matrices[layer])
        bar = '█' * int(cka_val * 40) + '░' * (40 - int(cka_val * 40))
        print(f'  L{layer:02d} vs L00: CKA={cka_val:.4f}  {bar}')
        evolution.append({'layer': layer, 'cka_vs_input': round(cka_val, 4)})
        if cka_val < 0.5 and transform_point is None:
            transform_point = layer
    print(f'  Transformation point (CKA < 0.5): {"Layer " + str(transform_point) if transform_point else "Not found"}')

    clustered = sum(1 for c in cluster_cka_by_layer if c['verdict'] == 'CLUSTERED')
    print()
    print('SUMMARY')
    print('=' * 80)
    print(f'  Layers with semantic clustering: {clustered}/{len(SAMPLE_LAYERS)}')
    print(f'  Transformation point: {"L" + str(transform_point) if transform_point else "N/A"}')

    out_file = f'{OUTPUT_DIR}/bol_cka_{LABEL}.json'
    with open(out_file, 'w') as f:
        json.dump({
            'label': LABEL, 'model': MODEL, 'num_layers': N, 'sampled_layers': SAMPLE_LAYERS,
            'cka_matrix': {str(k): v for k, v in cka_matrix.items()},
            'cluster_cka_by_layer': cluster_cka_by_layer,
            'representation_evolution': evolution,
            'transformation_point': transform_point,
        }, f, indent=2)
    print(f'Saved: {out_file}')

run_cka()

In [ ]:
# ── CELL 10: ATTENTION MAP — Attention Pattern Analysis ───────────────────────
def run_attention_map():
    def get_attention(word):
        inp = tokenizer(word, return_tensors='pt')
        if DEVICE == 'cuda':
            inp = {k: v.to(DEVICE) for k, v in inp.items()}
        with torch.no_grad():
            out = model(**inp, output_attentions=True)
        attn_per_layer = []
        for layer_attn in out.attentions:
            last_tok_attn = layer_attn[0, :, -1, :].cpu().float()
            attn_per_layer.append(last_tok_attn)
        return attn_per_layer

    def attention_entropy(attn_dist):
        p = attn_dist.clamp(min=1e-10)
        return -(p * p.log()).sum().item()

    def attention_pattern_similarity(attn_a, attn_b):
        sims = []
        for layer_a, layer_b in zip(attn_a, attn_b):
            min_heads = min(layer_a.shape[0], layer_b.shape[0])
            min_len = min(layer_a.shape[1], layer_b.shape[1])
            a = layer_a[:min_heads, :min_len]
            b = layer_b[:min_heads, :min_len]
            a = a / a.sum(dim=-1, keepdim=True).clamp(min=1e-10)
            b = b / b.sum(dim=-1, keepdim=True).clamp(min=1e-10)
            m = 0.5 * (a + b)
            kl_am = (a * (a / m.clamp(min=1e-10)).log()).sum(dim=-1)
            kl_bm = (b * (b / m.clamp(min=1e-10)).log()).sum(dim=-1)
            jsd = 0.5 * (kl_am + kl_bm)
            sims.append(1.0 - jsd.mean().item())
        return sims

    print('Extracting attention patterns for all words...')
    all_words = list({w for pair in RELATED_PAIRS + UNRELATED_PAIRS for w in pair})
    attn_cache = {w: get_attention(w) for w in all_words}
    print(f'  Cached {len(attn_cache)} words x {N} layers x {NUM_HEADS} heads')
    print()

    print('=' * 75)
    print('TEST 1: ATTENTION ENTROPY BY LAYER')
    print('(Low entropy = focused, High = uniform/lost)')
    print('=' * 75)
    print(f'{"Layer":>6}  {"mean_H":>8}  {"norm_H":>8}  {"min_H":>8}  {"max_H":>8}  bar')
    print('-' * 75)

    entropy_by_layer = []
    for layer in range(N):
        layer_entropies = []
        for word, attn_layers in attn_cache.items():
            attn = attn_layers[layer]
            seq_len = attn.shape[1]
            for head in range(attn.shape[0]):
                h = attention_entropy(attn[head])
                layer_entropies.append(h)
        mean_h = np.mean(layer_entropies)
        max_possible = math.log(seq_len) if seq_len > 0 else 0
        norm_h = mean_h / max_possible if max_possible > 0 else 0
        bar = '█' * int(norm_h * 40) + '░' * (40 - int(norm_h * 40))
        print(f'  L{layer:02d}   {mean_h:>8.4f}  {norm_h:>8.4f}  {min(layer_entropies):>8.4f}  {max(layer_entropies):>8.4f}  {bar}')
        entropy_by_layer.append({
            'layer': layer, 'mean_entropy': round(mean_h, 4), 'normalized_entropy': round(norm_h, 4),
            'min_entropy': round(min(layer_entropies), 4), 'max_entropy': round(max(layer_entropies), 4),
            'interpretation': 'focused' if norm_h < 0.5 else 'moderate' if norm_h < 0.8 else 'diffuse',
        })

    print()
    print('=' * 75)
    print('TEST 2: ATTENTION PATTERN SIMILARITY (Related vs Unrelated)')
    print('=' * 75)
    print(f'{"Layer":>6}  {"rel_sim":>9}  {"unrel_sim":>10}  {"sep":>8}  {"p-val":>8}  verdict')
    print('-' * 75)

    attn_separation = []
    for layer in range(N):
        rel_sims = [attention_pattern_similarity(attn_cache[a], attn_cache[b])[layer]
                    for a, b in RELATED_PAIRS]
        unrel_sims = [attention_pattern_similarity(attn_cache[a], attn_cache[b])[layer]
                      for a, b in UNRELATED_PAIRS]
        rel_avg, unrel_avg = np.mean(rel_sims), np.mean(unrel_sims)
        sep = rel_avg - unrel_avg
        try:
            _, pval = scipy_stats.mannwhitneyu(rel_sims, unrel_sims, alternative='greater')
        except ValueError:
            pval = 1.0
        verdict = ('STRUCTURED' if sep > 0.02 and pval < 0.05
                   else 'WEAK' if sep > 0 else 'NO STRUCTURE')
        sig = '*' if pval < 0.05 else ' '
        print(f'  L{layer:02d}   {rel_avg:>9.4f}  {unrel_avg:>10.4f}  {sep:>+8.4f}  {pval:>7.4f}{sig} {verdict}')
        attn_separation.append({
            'layer': layer, 'related_sim': round(rel_avg, 4), 'unrelated_sim': round(unrel_avg, 4),
            'separation': round(sep, 4), 'p_value': round(pval, 6),
            'significant': pval < 0.05, 'verdict': verdict,
        })

    print()
    print('=' * 75)
    print('TEST 3: HEAD SPECIALIZATION BY LAYER')
    print('(High = heads do different things, Low = redundant heads)')
    print('=' * 75)
    print(f'{"Layer":>6}  {"head_div":>10}  {"interpret":<15}')
    print('-' * 45)

    head_specialization = []
    for layer in range(N):
        head_patterns = [[] for _ in range(NUM_HEADS)]
        for word, attn_layers in attn_cache.items():
            attn = attn_layers[layer]
            for h in range(min(attn.shape[0], NUM_HEADS)):
                head_patterns[h].append(attn[h].numpy())
        avg_patterns = []
        for h in range(NUM_HEADS):
            if head_patterns[h]:
                max_len = max(p.shape[0] for p in head_patterns[h])
                padded = [np.pad(p, (0, max_len - p.shape[0])) for p in head_patterns[h]]
                avg_patterns.append(np.mean(padded, axis=0))
        if len(avg_patterns) >= 2:
            dists = []
            for i in range(len(avg_patterns)):
                for j in range(i + 1, len(avg_patterns)):
                    a = torch.tensor(avg_patterns[i]).unsqueeze(0)
                    b = torch.tensor(avg_patterns[j]).unsqueeze(0)
                    sim = F.cosine_similarity(a, b).item()
                    dists.append(1.0 - sim)
            diversity = np.mean(dists)
        else:
            diversity = 0.0
        interp = 'specialized' if diversity > 0.3 else 'moderate' if diversity > 0.1 else 'redundant'
        print(f'  L{layer:02d}   {diversity:>10.4f}  {interp:<15}')
        head_specialization.append({'layer': layer, 'head_diversity': round(diversity, 4), 'interpretation': interp})

    structured_layers = sum(1 for a in attn_separation if a['verdict'] == 'STRUCTURED')
    specialized_layers = sum(1 for h in head_specialization if h['interpretation'] == 'specialized')
    focused_layers = sum(1 for e in entropy_by_layer if e['interpretation'] == 'focused')
    print()
    print('=' * 75)
    print('SUMMARY')
    print('=' * 75)
    print(f'  Structured attention layers (rel > unrel, p<0.05): {structured_layers}/{N}')
    print(f'  Specialized head layers (diversity > 0.3):         {specialized_layers}/{N}')
    print(f'  Focused attention layers (norm entropy < 0.5):     {focused_layers}/{N}')

    out_file = f'{OUTPUT_DIR}/bol_attention_map_{LABEL}.json'
    with open(out_file, 'w') as f:
        json.dump({
            'label': LABEL, 'model': MODEL, 'num_layers': N, 'num_heads': NUM_HEADS,
            'entropy_by_layer': entropy_by_layer, 'attention_separation': attn_separation,
            'head_specialization': head_specialization,
            'structured_layers': structured_layers, 'specialized_layers': specialized_layers,
            'focused_layers': focused_layers,
        }, f, indent=2)
    print(f'Saved: {out_file}')

run_attention_map()

In [ ]:
# ── CELL 11: WEIGHT FINGERPRINT — Pre-Surgery Inspection ──────────────────────
def run_weight_fingerprint():
    # Add more models here for cross-subject comparison:
    # models_to_fingerprint = {LABEL: MODEL, 'subject_b': 'other/model'}
    models_to_fingerprint = {LABEL: MODEL}

    all_fingerprints = {}
    for label, path in models_to_fingerprint.items():
        print(f'=== FINGERPRINTING: {label} ===')
        print(f'{"Tensor":<55} {"mean":>10} {"std":>10} {"sparse":>8} {"spec_norm":>10}')
        print('-' * 95)
        fp_model = AutoModelForCausalLM.from_pretrained(path)
        fingerprint = []
        for name, param in fp_model.named_parameters():
            if '.layers.' not in name or param.ndim < 2:
                continue
            data = param.data.float()
            stats = {
                'name': name, 'shape': list(param.shape),
                'mean': round(data.mean().item(), 6), 'std': round(data.std().item(), 6),
                'abs_mean': round(data.abs().mean().item(), 6),
                'max': round(data.max().item(), 6), 'min': round(data.min().item(), 6),
                'sparsity': round((data.abs() < 1e-6).float().mean().item(), 4),
                'frobenius': round(data.norm().item(), 4),
            }
            if param.ndim == 2 and max(param.shape) <= 2048:
                try:
                    s = torch.linalg.svdvals(data)
                    stats['spectral_norm'] = round(s[0].item(), 4)
                    stats['condition_number'] = round((s[0] / (s[-1] + 1e-9)).item(), 4)
                except Exception:
                    stats['spectral_norm'] = stats['condition_number'] = None
            else:
                stats['spectral_norm'] = stats['condition_number'] = None
            sn = f'{stats["spectral_norm"]:.4f}' if stats['spectral_norm'] else '  n/a  '
            print(f'  {name:<53} {stats["mean"]:>10.6f} {stats["std"]:>10.6f} {stats["sparsity"]:>8.4f} {sn:>10}')
            fingerprint.append(stats)
        all_fingerprints[label] = fingerprint
        del fp_model
        print(f'  -> {len(fingerprint)} tensors fingerprinted for {label}')
        print()

    out_file = f'{OUTPUT_DIR}/weight_fingerprints_{LABEL}.json'
    with open(out_file, 'w') as f:
        json.dump(all_fingerprints, f, indent=2)
    print(f'Saved: {out_file}')

run_weight_fingerprint()

In [ ]:
# ── CELL 12: RANDOM BASELINE — Null Hypothesis Control ───────────────────────
def run_random_baseline():
    def get_layers_from_word(word):
        inp = tokenizer(word, return_tensors='pt')
        if DEVICE == 'cuda':
            inp = {k: v.to(DEVICE) for k, v in inp.items()}
        with torch.no_grad():
            out = model(**inp)
        return [h[0, -1, :].cpu().float().numpy() for h in out.hidden_states]

    def get_layers_from_random_id():
        random_id = torch.randint(0, VOCAB_SIZE, (1, 1))
        attention_mask = torch.ones_like(random_id)
        if DEVICE == 'cuda':
            random_id = random_id.to(DEVICE)
            attention_mask = attention_mask.to(DEVICE)
        with torch.no_grad():
            out = model(input_ids=random_id, attention_mask=attention_mask)
        return [h[0, -1, :].cpu().float().numpy() for h in out.hidden_states]

    def measure_separation(cache_a, cache_b, pairs):
        results = []
        for layer in range(N + 1):
            sims = []
            for key_a, key_b in pairs:
                a = torch.tensor(cache_a[key_a][layer]).unsqueeze(0)
                b = torch.tensor(cache_b[key_b][layer]).unsqueeze(0)
                sims.append(F.cosine_similarity(a, b).item())
            results.append({'layer': layer, 'sims': sims, 'mean': np.mean(sims)})
        return results

    print('=== CONDITION 1: Real words (ground truth) ===')
    all_words = list({w for pair in RELATED_PAIRS + UNRELATED_PAIRS for w in pair})
    real_cache = {w: get_layers_from_word(w) for w in all_words}
    real_rel   = measure_separation(real_cache, real_cache, RELATED_PAIRS)
    real_unrel = measure_separation(real_cache, real_cache, UNRELATED_PAIRS)

    print('=== CONDITION 2: Random token IDs (null hypothesis) ===')
    NUM_RANDOM = len(all_words) + 10
    random_cache = {f'rand_{i}': get_layers_from_random_id() for i in range(NUM_RANDOM)}
    random_keys = list(random_cache.keys())
    n_pairs = len(RELATED_PAIRS)
    random_pairs_a = [(random_keys[i], random_keys[i + n_pairs]) for i in range(n_pairs)]
    random_pairs_b = [(random_keys[i], random_keys[i + 2 * n_pairs]) for i in range(n_pairs)]
    rand_group_a = measure_separation(random_cache, random_cache, random_pairs_a)
    rand_group_b = measure_separation(random_cache, random_cache, random_pairs_b)

    print('=== CONDITION 3: Real words, shuffled pairings ===')
    np.random.seed(42)
    flat_words = list(all_words)
    np.random.shuffle(flat_words)
    n = len(RELATED_PAIRS)
    shuffled_rel   = [(flat_words[i], flat_words[i + n]) for i in range(n)]
    shuffled_unrel = [(flat_words[i], flat_words[i + 2 * n]) for i in range(min(n, len(flat_words) - 2 * n))]
    shuf_rel   = measure_separation(real_cache, real_cache, shuffled_rel)
    shuf_unrel = measure_separation(real_cache, real_cache, shuffled_unrel)

    print()
    print('=' * 90)
    print('SEPARATION COMPARISON: Real vs Random vs Shuffled')
    print('=' * 90)
    print(f'{"Layer":>6}  {"Real sep":>10}  {"p-val":>8}  {"Rand sep":>10}  {"p-val":>8}  {"Shuf sep":>10}  {"p-val":>8}')
    print('-' * 90)

    layer_comparison = []
    for layer in range(N + 1):
        real_r, real_u = real_rel[layer]['sims'], real_unrel[layer]['sims']
        real_sep = np.mean(real_r) - np.mean(real_u)
        rand_a, rand_b = rand_group_a[layer]['sims'], rand_group_b[layer]['sims']
        rand_sep = np.mean(rand_a) - np.mean(rand_b)
        shuf_r = shuf_rel[layer]['sims']
        shuf_u = shuf_unrel[layer]['sims'] if layer < len(shuf_unrel) else shuf_r
        shuf_sep = np.mean(shuf_r) - np.mean(shuf_u)
        try:
            _, real_p = scipy_stats.mannwhitneyu(real_r, real_u, alternative='greater')
        except ValueError:
            real_p = 1.0
        try:
            _, rand_p = scipy_stats.mannwhitneyu(rand_a, rand_b, alternative='greater')
        except ValueError:
            rand_p = 1.0
        try:
            _, shuf_p = scipy_stats.mannwhitneyu(shuf_r, shuf_u, alternative='greater')
        except ValueError:
            shuf_p = 1.0
        rs = '*' if real_p < 0.05 else ' '
        ras = '*' if rand_p < 0.05 else ' '
        ss = '*' if shuf_p < 0.05 else ' '
        print(f'  L{layer:02d}   {real_sep:>+10.4f}  {real_p:>7.4f}{rs} {rand_sep:>+10.4f}  {rand_p:>7.4f}{ras} {shuf_sep:>+10.4f}  {shuf_p:>7.4f}{ss}')
        layer_comparison.append({
            'layer': layer,
            'real_separation': round(real_sep, 4), 'real_p_value': round(real_p, 6), 'real_significant': real_p < 0.05,
            'random_separation': round(rand_sep, 4), 'random_p_value': round(rand_p, 6), 'random_significant': rand_p < 0.05,
            'shuffled_separation': round(shuf_sep, 4), 'shuffled_p_value': round(shuf_p, 6), 'shuffled_significant': shuf_p < 0.05,
        })

    real_sig = sum(1 for c in layer_comparison if c['real_significant'])
    rand_sig = sum(1 for c in layer_comparison if c['random_significant'])
    shuf_sig = sum(1 for c in layer_comparison if c['shuffled_significant'])
    print()
    print('INTERPRETATION')
    print('=' * 90)
    print(f'  Significant layers (p<0.05):')
    print(f'    Real word pairs:     {real_sig}/{N+1}')
    print(f'    Random token IDs:    {rand_sig}/{N+1}')
    print(f'    Shuffled pairings:   {shuf_sig}/{N+1}')
    if real_sig > 0 and rand_sig == 0 and shuf_sig <= 2:
        print('  PASS: Real pairs show significant separation, controls do not. Methodology is sound.')
        verdict = 'PASS'
    elif rand_sig > 2:
        print('  FAIL: Random tokens show significant separation. Possible systematic bias.')
        verdict = 'FAIL'
    else:
        print('  INCONCLUSIVE: Results do not clearly distinguish real from random.')
        verdict = 'INCONCLUSIVE'

    out_file = f'{OUTPUT_DIR}/bol_random_baseline_{LABEL}.json'
    with open(out_file, 'w') as f:
        json.dump({
            'label': LABEL, 'model': MODEL, 'num_layers': N, 'verdict': verdict,
            'sig_layers_real': real_sig, 'sig_layers_random': rand_sig, 'sig_layers_shuffled': shuf_sig,
            'layer_comparison': layer_comparison,
        }, f, indent=2)
    print(f'Saved: {out_file}')

run_random_baseline()

In [ ]:
# ── CELL 13: MULTIRUN STABILITY — Reproducibility Test ───────────────────────
def run_multirun_stability():
    N_RUNS = 10
    related   = RELATED_PAIRS[:6]
    unrelated = UNRELATED_PAIRS[:6]

    def get_all_layers(word):
        inp = tokenizer(word, return_tensors='pt')
        if DEVICE == 'cuda':
            inp = {k: v.to(DEVICE) for k, v in inp.items()}
        with torch.no_grad():
            out = model(**inp)
        return [h[0, -1, :].cpu().float() for h in out.hidden_states]

    def cos(a, b):
        return F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

    def measure_separation_fn():
        all_words = list({w for pair in related + unrelated for w in pair})
        cache = {w: get_all_layers(w) for w in all_words}
        seps = []
        for layer in range(N + 1):
            rel_sims   = [cos(cache[a][layer], cache[b][layer]) for a, b in related]
            unrel_sims = [cos(cache[a][layer], cache[b][layer]) for a, b in unrelated]
            seps.append(np.mean(rel_sims) - np.mean(unrel_sims))
        return seps

    def verify_hash():
        hasher = hashlib.sha256()
        for name, param in model.named_parameters():
            hasher.update(param.data.cpu().numpy().tobytes())
        return hasher.hexdigest()[:16]

    print('Saving original weight snapshot...')
    original_weights = {name: param.data.clone() for name, param in model.named_parameters()}
    baseline_hash = verify_hash()

    print('Computing baseline separation...')
    baseline_seps = measure_separation_fn()
    best_baseline = max(baseline_seps)
    best_baseline_layer = int(np.argmax(baseline_seps))
    print(f'  Baseline best separation: {best_baseline:+.4f} at layer {best_baseline_layer}')
    print()
    print(f'Running SHUFFLE test {N_RUNS} times with different random seeds...')
    print()

    all_run_seps = []
    for run in range(N_RUNS):
        seed = run * 42 + 7
        torch.manual_seed(seed)
        embed_names = []
        for name, param in model.named_parameters():
            if 'embed_tokens' in name:
                perm = torch.randperm(param.data.shape[0])
                param.data.copy_(original_weights[name][perm])
                embed_names.append(name)
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
        seps = measure_separation_fn()
        all_run_seps.append(seps)
        best_sep = max(seps)
        best_layer = int(np.argmax(seps))
        middle_survive = any(s > 0.05 for s, l in zip(seps, range(N + 1)) if 5 <= l <= N - 5)
        status = 'H1' if middle_survive else 'H2'
        print(f'  Run {run+1:02d} (seed={seed:4d}): best_sep={best_sep:+.4f} at L{best_layer:02d}  [{status}]')
        for name in embed_names:
            model.get_parameter(name).data.copy_(original_weights[name])

    final_hash = verify_hash()
    print()
    if final_hash == baseline_hash:
        print(f'INTEGRITY VERIFIED (Hash: {final_hash})')
    else:
        print(f'INTEGRITY FAIL!')

    all_run_seps = np.array(all_run_seps)
    print()
    print(f'STABILITY ANALYSIS ACROSS {N_RUNS} RUNS')
    print('=' * 85)
    print(f'{"Layer":>6}  {"baseline":>10}  {"mean":>8}  {"std":>8}  {"min":>8}  {"max":>8}  {"CV":>8}  stable?')
    print('-' * 85)

    stability_data = []
    for layer in range(N + 1):
        bl   = baseline_seps[layer]
        vals = all_run_seps[:, layer]
        mean = np.mean(vals)
        std  = np.std(vals)
        cv   = abs(std / mean) if abs(mean) > 1e-6 else float('inf')
        stable = 'YES' if cv < 0.5 else 'MARGINAL' if cv < 1.0 else 'NO'
        print(f'  L{layer:02d}   {bl:>+10.4f}  {mean:>+8.4f}  {std:>8.4f}  {min(vals):>+8.4f}  {max(vals):>+8.4f}  {cv:>8.2f}  {stable}')
        stability_data.append({
            'layer': layer, 'baseline_sep': round(bl, 4), 'mean_sep': round(mean, 4),
            'std_sep': round(std, 4), 'min_sep': round(min(vals), 4), 'max_sep': round(max(vals), 4),
            'cv': round(cv, 4) if cv != float('inf') else None, 'stable': stable,
        })

    h1_runs = sum(
        1 for run_seps in all_run_seps
        if any(s > 0.05 for s, l in zip(run_seps, range(N + 1)) if 5 <= l <= N - 5)
    )
    stable_layers = sum(1 for s in stability_data if s['stable'] == 'YES')
    best_layers   = [int(np.argmax(run)) for run in all_run_seps]
    mode_best     = max(set(best_layers), key=best_layers.count)
    mode_count    = best_layers.count(mode_best)

    print()
    print('OVERALL VERDICT')
    print('=' * 85)
    print(f'  Runs supporting H1 (octopus phase):       {h1_runs}/{N_RUNS}')
    print(f'  Runs supporting H2 (embedding-dependent): {N_RUNS - h1_runs}/{N_RUNS}')
    print(f'  Stable layers (CV < 0.5):                 {stable_layers}/{N+1}')
    print(f'  Most common best layer:                   L{mode_best:02d} ({mode_count}/{N_RUNS} runs)')

    if h1_runs == N_RUNS:
        reproducibility = 'FULLY_REPRODUCIBLE'
        print(f'  REPRODUCIBLE: H1 holds across all {N_RUNS} seeds.')
    elif h1_runs > N_RUNS * 0.8:
        reproducibility = 'MOSTLY_REPRODUCIBLE'
        print(f'  MOSTLY REPRODUCIBLE: H1 holds in {h1_runs}/{N_RUNS} runs.')
    elif h1_runs > N_RUNS * 0.5:
        reproducibility = 'MIXED'
        print(f'  MIXED: H1 holds in {h1_runs}/{N_RUNS} runs. Seed-sensitive.')
    else:
        reproducibility = 'NOT_REPRODUCIBLE'
        print(f'  NOT REPRODUCIBLE: H1 only holds in {h1_runs}/{N_RUNS} runs.')

    out_file = f'{OUTPUT_DIR}/bol_multirun_stability_{LABEL}.json'
    with open(out_file, 'w') as f:
        json.dump({
            'label': LABEL, 'model': MODEL, 'num_layers': N, 'n_runs': N_RUNS,
            'reproducibility': reproducibility, 'h1_runs': h1_runs, 'h2_runs': N_RUNS - h1_runs,
            'most_common_best_layer': int(mode_best), 'best_layer_consistency': f'{mode_count}/{N_RUNS}',
            'baseline_separations': [round(s, 4) for s in baseline_seps],
            'stability_by_layer': stability_data, 'all_run_best_layers': best_layers,
        }, f, indent=2)
    print(f'Saved: {out_file}')

run_multirun_stability()

In [ ]:
./build/bin/llama-cli \
  -m /root/.cache/huggingface/hub/models--juiceb0xc0de--moomoo/snapshots/1f8c44e83a9ebc36c3498c7e0b4fedf940059ef6 \
  -ngl 99 \
  --temp 0.9 \
  --min-p 0.1 \
  --top-p 1.0 \
  --top-k 0 \
  --repeat-penalty 1.1 \
  --repeat-last-n 256 \
  -c 8192 \
  -p "<|im_start|>system
You are Bella. No asterisk actions. No AI disclaimers. No bullet points. No 'great question' openers.<|im_end|>
<|im_start|>user
Hey.<|im_end|>
<|im_start|>assistant
" \
  -cnv

In [ ]:
cd /desktop/model.safetensors \
python convert_hf_to_gguf.py \
  /workspace/bella-7794-merged \
  --outfile /workspace/bella-7794-f16.gguf \
  --outtype f16

In [ ]:
exit
